In [ ]:
import numpy as np 
import pymtrf 
import matplotlib.pyplot as plt
import seaborn as sns
import os
import librosa
from scipy import interpolate
from scipy.signal import resample as sci_resample
import glob
import mne
from eelbrain import *

In [ ]:
def rms_interpolate(audio_file, eeg_sr=128, eeg_duration=[13, 103]):

    stim, sr = librosa.load(audio_file)

    # Compute RMS 
    rms_win = 0.01 # 10ms
    rms_hop = 1/eeg_sr # hop by eeg sampling rate
    rms = librosa.feature.rms(y=stim,
                            frame_length=int(sr*rms_win),
                            hop_length=int(sr*rms_hop))
    rms_sr = 1/rms_hop # the rms time series is sampled with period rms_hop
    rms=rms[0]

    # resample to same rate as EEG
    rms_resampled = sci_resample(rms, int(eeg_sr*len(rms)/rms_sr))
    rms_resampled=rms_resampled[:eeg_duration[1]]

    # pad 
    pad_before = eeg_duration[0]
    pad_after = eeg_duration[1] - (len(rms_resampled)+pad_before)
    rms_padded = np.pad(rms_resampled, pad_width=(pad_before,pad_after))
    rms_padded = rms_padded.astype('<f8')
    rms_padded = np.where(np.isfinite(rms_padded), rms_padded, 0)
    
    return rms_padded

In [ ]:
rms_std = rms_interpolate('./sounds/standard_1000Hz_300ms_15dB_sigmoid_0_flat_0ms_slope_7.wav')
rms_looming = rms_interpolate('./sounds/deviant_looming_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav')
rms_receding = rms_interpolate('./sounds/deviant_receding_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav')
rms_flat = rms_interpolate('./sounds/deviant_flat_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav')
# rms_std_log = (rms_flat-rms_std) ** (1./3)
rms_looming_log = np.sign(rms_looming-rms_std) * (np.abs(rms_looming-rms_std)) ** (1 / 3)
rms_receding_log = np.sign(rms_receding-rms_std) * (np.abs(rms_receding-rms_std)) ** (1 / 3)
rms_flat_log = np.sign(rms_flat-rms_std) * (np.abs(rms_flat-rms_std)) ** (1 / 3)

times_long = np.arange(-0.100, 0.701, 1/128)

fig, ax = plt.subplots(nrows=4, sharex=True, sharey=True, figsize=(8, 8))
fig.tight_layout(pad=5)
ax[0].plot(times_long, rms_std)
ax[0].set_title('Standard RMS')
ax[1].plot(times_long, rms_looming)
ax[1].set_title('Looming RMS')
ax[2].plot(times_long, rms_receding)
ax[2].set_title('Receding RMS')
ax[3].plot(times_long, rms_flat)
ax[3].set_title('Flat RMS')

fig, ax = plt.subplots(nrows=3, sharex=True, sharey=True, figsize=(8, 8))
fig.tight_layout(pad=5)
ax[0].plot(times_long, rms_looming_log)
ax[0].set_title('Looming-Standard RMS')
ax[1].plot(times_long, rms_receding_log)
ax[1].set_title('Receding-Standard RMS')
ax[2].plot(times_long, rms_flat_log)
ax[2].set_title('Flat-Standard RMS')

In [ ]:
# stim_dict = {
#                 '1002': rms_interpolate('./sounds/deviant_looming_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav') - rms_interpolate('./sounds/standard_1000Hz_300ms_15dB_sigmoid_0_flat_0ms_slope_7.wav'), 
#                 '1003': rms_interpolate('./sounds/deviant_receding_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav') - rms_interpolate('./sounds/standard_1000Hz_300ms_15dB_sigmoid_0_flat_0ms_slope_7.wav'), 
#                 '1004': rms_interpolate('./sounds/deviant_flat_1000Hz_600ms_15dB_sigmoid_0_flat_0ms_slope_7.wav') - rms_interpolate('./sounds/standard_1000Hz_300ms_15dB_sigmoid_0_flat_0ms_slope_7.wav')
#             }

stim_dict = {
                '1002': rms_looming_log,
                '1003': rms_receding_log,
                '1004': rms_flat_log
            }

In [ ]:
subjs_all = glob.glob('./analysis_800ms_epochs/looming*-epo.fif')
subjs = []
for i in subjs_all:
    subjs.append(i.lower())

subjs = sorted(subjs, key=lambda x: int(x.split('-')[0][-3:]))
del subjs[9]
del subjs[0]
print(subjs)

#### Compute TRF and predict Fz for each subject

In [ ]:
actuals = []
predictions = []

In [ ]:
for subj in subjs:
    standard_evks = []
    looming_evks = []
    receding_evks = []
    flat_evks = []

    eeg_data = []
    envelopes = []

    # actuals_subj = []
    # predictions_subj = []

    subject = int(subj.split('-')[0][-3:])
    subj_epochs = mne.read_epochs(subj, preload=True, verbose=False)
    subj_epochs = subj_epochs.drop_channels('STI')
    subj_epochs.resample(128)

    standard_evks.append(subj_epochs['1001'].average())
    # looming_evks.append(subj_epochs['1002'].average())
    receding_evks.append(subj_epochs['1003'].average())
    # flat_evks.append(subj_epochs['1004'].average())

    for i in range(len(subj_epochs)):
        if subj_epochs[i].events[:, 2][0] != 1001 and subj_epochs[i].events[:, 2][0] != 1002 and subj_epochs[i].events[:, 2][0] != 1003:
            eeg_deviant = subj_epochs[i].get_data()[0]
            eeg_std = subj_epochs[i-1].get_data()[0]
            eeg = (eeg_deviant - eeg_std).T
            eeg_data.append(eeg)

            envelope = stim_dict.get(str(subj_epochs[i].events[:, 2][0]))
            envelope = np.reshape(envelope, (-1, 1))
            envelopes.append(envelope)


    eeg_data = np.mean(np.asarray(eeg_data), axis=0)
    envelopes = np.mean(np.asarray(envelopes), axis=0)
    # eeg_data = np.asarray(eeg_data)
    # envelopes = np.asarray(envelopes) 
    print(eeg_data.shape)
    print(envelopes.shape)
    [w, t, i] = pymtrf.mtrf_train(envelopes, eeg_data, 128, 1, -100, 701, 1)
        
    stim_test = np.asarray([rms_receding-rms_std]).T

    receding_combined = mne.combine_evoked(receding_evks, weights='nave')
    standard_combined = mne.combine_evoked(standard_evks, weights='nave')
    receding_standard = mne.combine_evoked([receding_combined, standard_combined], weights=[1, -1])
    resp_test = receding_standard.data.T
    Fz_for_comp = receding_standard.pick('Fz').data.T
    peak_actual = np.amin(Fz_for_comp[63:79], axis=0)
    actuals.append(Fz_for_comp)

    [prediction, r, p, mse] = pymtrf.mtrf_predict(stim_test, resp_test, w, 128, 1, -100, 701, i)
    peak_pred = np.amin(prediction[:, 1][63:79], axis=0)
    scaling_factor = peak_pred/peak_actual
    predictions.append(prediction[:, 1]/scaling_factor)
 
    plt.figure()
    plt.plot(times_long, Fz_for_comp)
    plt.plot(times_long, prediction[:, 1]/scaling_factor)
    plt.title(f'Subject {subject}')
    plt.gca().invert_yaxis()
    plt.show()

    # actuals.append(actuals_subj)
    # predictions.append(predictions_subj)

print(len(actuals))
print(len(predictions))

In [ ]:
# for j in range(len(actuals)):
#     plt.figure()
#     plt.plot(times_long, actuals[j])
#     plt.plot(times_long, predictions[j])
#     plt.axhline(y=0, color='k', linestyle='--')
#     plt.title(f"Subject {int(subjs[j].split('-')[0][-3:])}")
#     plt.gca().invert_yaxis()
#     plt.show()

In [ ]:
rows = []

tstep = 1. / 128
n_times = 103
time = UTS(-0.100, tstep, n_times)
sensor = Sensor.from_montage('easycap-M1')[:1]

for i in range(len(actuals)):
    subject = int(subjs[i].split('-')[0][-3:])
    condition = 'actual'
    fz_data = NDVar(actuals[i].flatten(), (time,))
    rows.append([subject, condition, fz_data])

for j in range(len(predictions)):
    subject = int(subjs[j].split('-')[0][-3:])
    condition = 'predicted'
    fz_data = NDVar(predictions[j], (time,))
    rows.append([subject, condition, fz_data])

ds = Dataset.from_caselist(['subject','condition', 'fz_data'], rows)
ds['subject'].random = True
print(ds.summary())

In [ ]:
res_timecoure = testnd.TTestRelated(
    'fz_data', 'condition', 'actual', 'predicted', match='subject', ds=ds,
    pmin=0.05,  # Use uncorrected p = 0.05 as threshold for forming clusters
    tstart=-0.100,  # Find clusters in the time window from 100 ...
    tstop=0.701,  # ... to 600 ms
    # mintime=0.1
)
clusters_fz = res_timecoure.find_clusters(0.05)
print(clusters_fz)

p = plot.UTSStat('fz_data', 'condition', match='subject', ds=ds, title='Compare conditions (Fz)', w=6, h=5)
p.set_clusters(clusters_fz)
p.set_ylim([2e-6, -9e-6])

In [ ]:
# % Plot CV accuracy
# figure
# subplot(2,2,1), errorbar(1:numel(lambda),mean(cv.r),std(cv.r)/sqrt(nfold-1),'linewidth',2)
# set(gca,'xtick',1:nlambda,'xticklabel',-6:2:6), xlim([0,numel(lambda)+1]), axis square, grid on
# title('CV Accuracy'), xlabel('Regularization (1\times10^\lambda)'), ylabel('Correlation')

# % Plot CV error
# subplot(2,2,2), errorbar(1:numel(lambda),mean(cv.err),std(cv.err)/sqrt(nfold-1),'linewidth',2)
# set(gca,'xtick',1:nlambda,'xticklabel',-6:2:6), xlim([0,numel(lambda)+1]), axis square, grid on
# title('CV Error'), xlabel('Regularization (1\times10^\lambda)'), ylabel('MSE')

# % Plot reconstruction
# subplot(2,2,3), plot((1:length(stest))/fs,stest,'linewidth',2), hold on
# plot((1:length(pred))/fs,pred,'linewidth',2), hold off, xlim([0,10]), axis square, grid on
# title('Reconstruction'), xlabel('Time (s)'), ylabel('Amplitude (a.u.)'), legend('Orig','Pred')

# % Plot test accuracy
# subplot(2,2,4), bar(1,rmax), hold on, bar(2,test.r), hold off
# set(gca,'xtick',1:2,'xticklabel',{'Val.','Test'}), axis square, grid on
# title('Model Performance'), xlabel('Dataset'), ylabel('Correlation')

In [ ]:
# % Plot accuracy
# figure
# subplot(1,2,1), h = fill(xacc,yacc,'b','edgecolor','none'); hold on
# set(h,'facealpha',0.2), xlim([tmin,tmax]), axis square, grid on
# plot(-fliplr(t),fliplr(macc),'linewidth',2), hold off
# title('Reconstruction Accuracy'), xlabel('Time lag (ms)'), ylabel('Correlation')

# % Plot error
# subplot(1,2,2)
# h = fill(xerr,yerr,'b','edgecolor','none'); hold on
# set(h,'facealpha',0.2), xlim([tmin,tmax]), axis square, grid on
# plot(-fliplr(t),fliplr(merr),'linewidth',2), hold off
# title('Reconstruction Error'), xlabel('Time lag (ms)'), ylabel('MSE')